In [ ]:
!pip install pandas numpy

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving customers.csv to customers.csv
Saving employees.csv to employees.csv
Saving order_items.csv to order_items.csv
Saving orders.csv to orders.csv
Saving products.csv to products.csv
Saving returns.csv to returns.csv
Saving stores.csv to stores.csv
Saving targets.csv to targets.csv


In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("customers.csv")
employees = pd.read_csv("employees.csv")
order_items = pd.read_csv("order_items.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
returns = pd.read_csv("returns.csv")
stores = pd.read_csv("stores.csv")
targets = pd.read_csv("targets.csv")

print("All 8 datasets loaded successfully!")

All 8 datasets loaded successfully!


In [ ]:
for df in [orders, employees, stores, targets]:
    if "Region" in df.columns:
        df["Region"] = df["Region"].astype("string").str.strip().str.title()

products["Category"] = products["Category"].astype("string").str.strip().str.title()
returns["Reason"] = returns["Reason"].astype("string").str.strip().str.title()

orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], errors="coerce")
returns["ReturnDate"] = pd.to_datetime(returns["ReturnDate"], errors="coerce")
targets["Month"] = pd.to_datetime(targets["Month"], errors="coerce")

print("Data cleaning completed!")

Data cleaning completed!


In [ ]:
items = (
    order_items
    .merge(
        products[["ProductID", "ProductName", "Category", "Price", "Cost"]],
        on="ProductID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        orders[[
            "OrderID", "CustomerID", "OrderDate", "Region", "StoreID",
            "EmployeeID", "SalesChannel", "PaymentMethod", "OrderStatus"
        ]],
        on="OrderID",
        how="left",
        validate="many_to_one"
    )
)

items["Revenue"] = items["Quantity"] * items["Price"] * (1 - items["Discount"])
items["CostAmount"] = items["Quantity"] * items["Cost"]
items["Profit"] = items["Revenue"] - items["CostAmount"]

valid_status = ["Completed", "Shipped"]

sales_items = items[items["OrderStatus"].isin(valid_status)].copy()
sales_orders = orders[orders["OrderStatus"].isin(valid_status)].copy()

order_sales = sales_items.groupby("OrderID", as_index=False).agg(
    Revenue=("Revenue", "sum"),
    Profit=("Profit", "sum"),
    QuantitySold=("Quantity", "sum")
)

order_detail = (
    sales_orders
    .merge(order_sales, on="OrderID", how="left", validate="one_to_one")
    .fillna({"Revenue": 0, "Profit": 0, "QuantitySold": 0})
)

print("Revenue and profit calculations completed!")

Revenue and profit calculations completed!


In [ ]:
q1 = sales_items[["Revenue", "Profit", "Quantity"]].sum()

print("Q1 - COMPANY TOTALS")
print(q1)

Q1 - COMPANY TOTALS
Revenue     5.112812e+08
Profit      2.640349e+08
Quantity    5.007300e+04
dtype: float64


In [ ]:
q2 = (
    order_detail
    .merge(customers[["CustomerID", "Name"]], on="CustomerID", how="left")
    [["OrderID", "CustomerID", "Name", "Region", "Revenue"]]
    .sort_values("Revenue", ascending=False)
    .head(10)
)

print("Q2 - TOP 10 ORDERS")
display(q2)

Q2 - TOP 10 ORDERS


,OrderID,CustomerID,Name,Region,Revenue
6449,O007675,C0085,Ananya Sharma,South,239507.1200
7225,O008617,C0786,Aditya Gupta,South,225928.1925
5890,O007022,C0851,Krishna Reddy,South,225831.7500
8011,O009560,C0183,Vihaan Verma,South,223284.9000
3597,O004265,C0932,Akash Gupta,East,214211.1500
2356,O002788,C0951,Ira Verma,West,211264.6000
6030,O007192,C0321,Rahul Khan,North,210803.9900
2132,O002513,C0373,Rohan Gupta,North,210304.7075
7816,O009324,C0285,Sneha Joshi,North,205302.1120
7434,O008862,C0212,Neha Joshi,East,202071.6750


In [ ]:
monthly_revenue = (
    sales_items.assign(
        Month=sales_items["OrderDate"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("Month", as_index=False)["Revenue"]
    .sum()
    .sort_values("Month")
)

print("Q3 - MONTHLY REVENUE")
display(monthly_revenue)

print("Highest month:")
display(monthly_revenue.loc[monthly_revenue["Revenue"].idxmax()])

print("Lowest month:")
display(monthly_revenue.loc[monthly_revenue["Revenue"].idxmin()])

Q3 - MONTHLY REVENUE


,Month,Revenue
0,2022-01-01,1.025533e+07
1,2022-02-01,8.449240e+06
2,2022-03-01,9.541239e+06
3,2022-04-01,7.390468e+06
4,2022-05-01,8.627147e+06
5,2022-06-01,7.203081e+06
6,2022-07-01,8.690864e+06
7,2022-08-01,7.574649e+06
8,2022-09-01,8.132531e+06
9,2022-10-01,8.720783e+06


Highest month:


,31
Month,2024-08-01 00:00:00
Revenue,10284478.3985


Lowest month:


,49
Month,2026-02-01 00:00:00
Revenue,6034845.1855


In [ ]:
q4 = (
    sales_items.groupby("Region", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
)

q4["RevenueSharePct"] = q4["Revenue"] / q4["Revenue"].sum() * 100

print("Q4 - REGION PERFORMANCE")
display(q4)

Q4 - REGION PERFORMANCE


,Region,Revenue,RevenueSharePct
2,South,1.493869e+08,29.218144
1,North,1.298550e+08,25.397956
3,West,1.252599e+08,24.499211
0,East,1.067795e+08,20.884689


In [ ]:
q5 = (
    sales_items.groupby(
        ["ProductID", "ProductName", "Category"], as_index=False
    )
    .agg(
        QuantitySold=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .sort_values("Revenue", ascending=False)
    .head(10)
)

print("Q5 - TOP 10 PRODUCTS")
display(q5)

Q5 - TOP 10 PRODUCTS


,ProductID,ProductName,Category,QuantitySold,Revenue,Profit
127,P128,Premium Backpack 128,Grocery,279,6341740.980,3057216.270
166,P167,Pro T-Shirt 167,Sports,298,6328047.912,5244805.992
131,P132,Ultra Cookware 132,Clothing,275,6117214.272,2046507.522
61,P062,Premium Chair 062,Sports,283,5911241.718,3213334.798
136,P137,Eco Chair 137,Toys,316,5877130.932,2540414.252
156,P157,Eco Cookware 157,Sports,255,5780748.060,4020059.760
113,P114,Smart T-Shirt 114,Beauty,289,5695021.629,931688.949
146,P147,Pro Headphones 147,Home & Kitchen,270,5661341.868,3862061.868
102,P103,Classic Mixer 103,Sports,268,5503805.380,4997941.980
197,P198,Eco Bottle 198,Clothing,256,5479014.420,3646243.860


In [ ]:
q6_categories = (
    sales_items.groupby("Category", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .sort_values("Profit", ascending=False)
    .head(5)
)

q6_low_products = (
    sales_items.groupby(["ProductID", "ProductName"], as_index=False)["Profit"]
    .sum()
    .sort_values("Profit")
    .head(5)
)

print("Q6 - TOP 5 CATEGORIES BY PROFIT")
display(q6_categories)

print("Q6 - LOWEST-PROFIT PRODUCTS")
display(q6_low_products)

Q6 - TOP 5 CATEGORIES BY PROFIT


,Category,Revenue,Profit
5,Home & Kitchen,6.640685e+07,3.839908e+07
6,Sports,6.542899e+07,3.812485e+07
2,Clothing,7.666034e+07,3.667200e+07
3,Electronics,5.897849e+07,3.504408e+07
4,Grocery,6.574442e+07,3.234569e+07


Q6 - LOWEST-PROFIT PRODUCTS


,ProductID,ProductName,Profit
153,P154,Everyday Lamp 154,-4.318310e+06
26,P027,Premium T-Shirt 027,-4.064260e+06
174,P175,Premium Chair 175,-3.301766e+06
135,P136,Smart Backpack 136,-1.338978e+06
54,P055,Smart Cookware 055,-5.007604e+04


In [ ]:
product_summary = (
    sales_items.groupby(
        ["ProductID", "ProductName", "Category"], as_index=False
    )
    .agg(
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
)

product_summary["ProfitMarginPct"] = np.where(
    product_summary["Revenue"] != 0,
    product_summary["Profit"] / product_summary["Revenue"] * 100,
    np.nan
)

q7 = (
    product_summary[product_summary["ProfitMarginPct"] < 10]
    .sort_values("ProfitMarginPct")
)

print("Q7 - PRODUCTS BELOW 10% MARGIN")
display(q7)

Q7 - PRODUCTS BELOW 10% MARGIN


,ProductID,ProductName,Category,Revenue,Profit,ProfitMarginPct
54,P055,Smart Cookware 055,Sports,1.800570e+06,-50076.0375,-2.781122
63,P064,Compact Cookware 064,Beauty,1.267270e+06,-29781.4130,-2.350045
42,P043,Classic Lamp 043,Electronics,9.840282e+05,-11093.3200,-1.127338
80,P081,Eco Backpack 081,Beauty,1.956176e+06,22044.1535,1.126900
149,P150,Everyday Headphones 150,Sports,1.669200e+06,21368.4345,1.280160
184,P185,Classic Mixer 185,Home & Kitchen,1.881235e+05,2904.6325,1.544003
145,P146,Compact Chair 146,Clothing,7.802455e+05,12928.1595,1.656935
3,P004,Classic Shoes 004,Grocery,9.116985e+05,17127.3000,1.878614
128,P129,Premium Notebook 129,Books,1.365965e+06,27598.1760,2.020415
175,P176,Smart Cookware 176,Grocery,2.273704e+06,52355.8940,2.302670


In [ ]:
employee_perf = (
    sales_items.groupby("EmployeeID", as_index=False)
    .agg(
        QuantitySold=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .merge(
        sales_orders.groupby("EmployeeID")
        .agg(OrdersHandled=("OrderID", "nunique"))
        .reset_index(),
        on="EmployeeID",
        how="outer"
    )
    .merge(
        employees[["EmployeeID", "Name", "Department", "Region"]],
        on="EmployeeID",
        how="left"
    )
)

employee_perf[
    ["QuantitySold", "Revenue", "Profit", "OrdersHandled"]
] = employee_perf[
    ["QuantitySold", "Revenue", "Profit", "OrdersHandled"]
].fillna(0)

q8 = employee_perf.sort_values(
    "Revenue", ascending=False
).head(10)

print("Q8 - TOP 10 EMPLOYEES")
display(q8)

Q8 - TOP 10 EMPLOYEES


,EmployeeID,QuantitySold,Revenue,Profit,OrdersHandled,Name,Department,Region
7,E008,617,6.606077e+06,3.583746e+06,97,Aditya Sharma,Marketing,West
3,E004,622,6.434383e+06,3.466271e+06,100,Sneha Pillai,Customer Support,West
23,E024,604,6.040254e+06,2.738165e+06,97,Amit Singh,Finance,North
45,E046,565,6.021649e+06,2.932976e+06,95,Akash Verma,Marketing,West
39,E040,584,5.995391e+06,3.075189e+06,96,Ira Gupta,Operations,East
48,E049,553,5.974522e+06,3.353761e+06,88,Pooja Bose,Marketing,South
83,E084,536,5.900048e+06,3.225632e+06,90,Akash Shetty,Customer Support,North
21,E022,584,5.891470e+06,2.922665e+06,102,Vivaan Mishra,IT,North
72,E073,572,5.873778e+06,2.901625e+06,97,Myra Menon,sales,East
14,E015,542,5.821239e+06,3.183237e+06,91,Neha Singh,IT,North


In [ ]:
print("Q9 - TOP REVENUE EMPLOYEE")
display(employee_perf.nlargest(1, "Revenue"))

print("Q9 - EMPLOYEE WITH MOST ORDERS")
display(employee_perf.nlargest(1, "OrdersHandled"))

Q9 - TOP REVENUE EMPLOYEE


,EmployeeID,QuantitySold,Revenue,Profit,OrdersHandled,Name,Department,Region
7,E008,617,6606076.562,3583746.042,97,Aditya Sharma,Marketing,West


Q9 - EMPLOYEE WITH MOST ORDERS


,EmployeeID,QuantitySold,Revenue,Profit,OrdersHandled,Name,Department,Region
21,E022,584,5891469.722,2922665.222,102,Vivaan Mishra,IT,North


In [ ]:
q10 = (
    sales_items.groupby(
        ["EmployeeID", "ProductID", "ProductName", "Category"],
        as_index=False
    )
    .agg(
        QuantitySold=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum")
    )
    .merge(
        employees[["EmployeeID", "Name"]],
        on="EmployeeID",
        how="left"
    )
    [[
        "EmployeeID", "Name", "ProductID", "ProductName",
        "Category", "QuantitySold", "Revenue", "Profit"
    ]]
    .sort_values(["Name", "Revenue"], ascending=[True, False])
)

print("Q10 - EMPLOYEE PRODUCT SALES")
display(q10.head(30))

Q10 - EMPLOYEE PRODUCT SALES


,EmployeeID,Name,ProductID,ProductName,Category,QuantitySold,Revenue,Profit
3100,E028,Aadhya Khan,P042,Pro Mixer 042,Toys,12,208468.0125,142866.8925
3144,E028,Aadhya Khan,P119,Everyday Chair 119,Home & Kitchen,11,208353.3870,171444.8670
3142,E028,Aadhya Khan,P114,Smart T-Shirt 114,Beauty,10,191751.5700,26930.3700
3179,E028,Aadhya Khan,P171,Everyday T-Shirt 171,Home & Kitchen,8,167597.5015,123093.4215
3191,E028,Aadhya Khan,P194,Pro Backpack 194,Toys,11,160325.3630,110232.5730
3094,E028,Aadhya Khan,P034,Ultra Notebook 034,Home & Kitchen,11,155997.0720,138078.6220
3186,E028,Aadhya Khan,P187,Eco Mixer 187,Electronics,7,130120.2000,128230.1300
3127,E028,Aadhya Khan,P089,Smart Chair 089,Grocery,7,122049.7110,115154.7810
3097,E028,Aadhya Khan,P039,Ultra T-Shirt 039,Electronics,7,119218.5510,78063.4510
3083,E028,Aadhya Khan,P011,Smart Mixer 011,Beauty,7,107668.0800,18557.5200


In [ ]:
q11 = (
    sales_items.groupby(
        ["Category", "EmployeeID"], as_index=False
    )["Revenue"]
    .sum()
    .merge(
        employees[["EmployeeID", "Name"]],
        on="EmployeeID",
        how="left"
    )
    .sort_values(["Category", "Revenue"], ascending=[True, False])
    .groupby("Category", as_index=False)
    .head(1)
    .sort_values("Category")
)

print("Q11 - BEST EMPLOYEE BY CATEGORY")
display(q11)

Q11 - BEST EMPLOYEE BY CATEGORY


,Category,EmployeeID,Revenue,Name
77,Beauty,E078,1.348479e+06,Rohan Nair
152,Books,E053,8.919211e+05,Meera Kumar
214,Clothing,E015,1.359012e+06,Neha Singh
372,Electronics,E073,1.308372e+06,Myra Menon
421,Grocery,E022,1.192495e+06,Vivaan Mishra
503,Home & Kitchen,E004,1.130817e+06,Sneha Pillai
694,Sports,E095,1.141782e+06,Shaurya Singh
703,Toys,E004,1.246098e+06,Sneha Pillai


In [ ]:
customer_spend = (
    order_detail.groupby("CustomerID", as_index=False)
    .agg(
        Orders=("OrderID", "nunique"),
        TotalRevenue=("Revenue", "sum")
    )
    .merge(
        customers[[
            "CustomerID", "Name", "City", "CustomerSegment"
        ]],
        on="CustomerID",
        how="left"
    )
)

q12 = customer_spend.sort_values(
    "TotalRevenue", ascending=False
).head(10)

print("Q12 - TOP 10 CUSTOMERS")
display(q12)

Q12 - TOP 10 CUSTOMERS


,CustomerID,Orders,TotalRevenue,Name,City,CustomerSegment
424,C0426,20,1.575923e+06,Nikhil Jain,Gurugram,New
495,C0497,17,1.137272e+06,Reyansh Rao,Mumbai,Regular
665,C0667,14,1.128175e+06,Aadhya Jain,Gurugram,Loyal
229,C0231,19,1.124779e+06,Kiran Kumar,CHANDIGARH,Loyal
894,C0896,12,1.122258e+06,Riya Pillai,Mumbai,Regular
412,C0414,18,1.102397e+06,Rahul Pillai,Ahmedabad,Loyal
182,C0184,15,1.099184e+06,Pari Singh,Ahmedabad,Regular
556,C0558,13,1.098173e+06,Reyansh Shetty,Bengaluru,Loyal
219,C0221,13,1.079469e+06,Nikhil Singh,Mangaluru,Loyal
151,C0152,14,1.067300e+06,Aarav Pillai,Patna,New


In [ ]:
q13 = (
    customer_spend.groupby("CustomerSegment", as_index=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        Revenue=("TotalRevenue", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

print("Q13 - CUSTOMER SEGMENTS")
display(q13)

Q13 - CUSTOMER SEGMENTS


,CustomerSegment,Customers,Revenue
3,Regular,457,2.386086e+08
1,Loyal,289,1.439498e+08
2,New,204,1.053420e+08
0,Corporate,49,2.280711e+07


In [ ]:
q14 = customers[
    ~customers["CustomerID"].isin(orders["CustomerID"])
][["CustomerID", "Name", "City", "CustomerSegment"]]

print("Q14 - CUSTOMERS WITHOUT ORDERS")
display(q14)

Q14 - CUSTOMERS WITHOUT ORDERS


,CustomerID,Name,City,CustomerSegment


In [ ]:
q15 = (
    order_detail.groupby("StoreID", as_index=False)
    .agg(
        Orders=("OrderID", "nunique"),
        Revenue=("Revenue", "sum")
    )
    .merge(
        stores[[
            "StoreID", "StoreName", "City", "Region", "StoreType"
        ]],
        on="StoreID",
        how="left"
    )
    .sort_values("Revenue", ascending=False)
)

print("Q15 - STORE PERFORMANCE")
display(q15)

Q15 - STORE PERFORMANCE


,StoreID,Orders,Revenue,StoreName,City,Region,StoreType
9,S010,485,3.106518e+07,Kochi Market,Kochi,South,Mall
5,S006,491,3.089530e+07,Bengaluru Central,Bengaluru,South,Shopping Complex
6,S007,498,2.952427e+07,Mangaluru Market,Mangaluru,South,High Street
8,S009,489,2.908515e+07,Hyderabad Mall,Hyderabad,South,Standalone
7,S008,473,2.867516e+07,CHENNAI MARKET,Chennai,South,Shopping Complex
3,S004,437,2.717065e+07,Lucknow Central,Lucknow,North,Shopping Complex
16,S017,448,2.668264e+07,Pune Mall,Pune,West,Mall
4,S005,428,2.653826e+07,Gurugram City,Gurugram,North,Shopping Complex
18,S019,426,2.651247e+07,Surat Mall,Surat,West,Mall
0,S001,428,2.637162e+07,Delhi Central,Delhi,North,Mall


In [ ]:
q16 = (
    order_detail.groupby("SalesChannel", as_index=False)
    .agg(
        Orders=("OrderID", "nunique"),
        Revenue=("Revenue", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

print("Q16 - SALES CHANNELS")
display(q16)

Q16 - SALES CHANNELS


,SalesChannel,Orders,Revenue
1,Online,4059,2.485335e+08
0,In-Store,3609,2.207621e+08
2,Phone,719,4.198566e+07


In [ ]:
q17_total = returns["OrderID"].nunique()

q17_reasons = (
    returns["Reason"]
    .astype("string")
    .str.strip()
    .str.title()
    .value_counts()
)

print("Q17 - UNIQUE RETURNED ORDERS:", q17_total)
print("RETURN REASONS")
display(q17_reasons)

Q17 - UNIQUE RETURNED ORDERS: 1000
RETURN REASONS


,count
Reason,
Late Delivery,169
Defective,168
Wrong Item,167
Changed Mind,165
Not As Expected,163
Damaged,158


In [ ]:
returned_order_region = (
    returns[["OrderID"]]
    .drop_duplicates()
    .merge(
        orders[["OrderID", "Region", "OrderStatus"]],
        on="OrderID",
        how="left"
    )
)

returned_counts = (
    returned_order_region[
        returned_order_region["OrderStatus"].isin(valid_status)
    ]
    .groupby("Region")["OrderID"]
    .nunique()
)

eligible_counts = (
    sales_orders.groupby("Region")["OrderID"].nunique()
)

q18 = pd.concat(
    [
        eligible_counts.rename("EligibleOrders"),
        returned_counts.rename("ReturnedOrders")
    ],
    axis=1
).fillna(0).reset_index()

q18["ReturnRatePct"] = (
    q18["ReturnedOrders"] / q18["EligibleOrders"] * 100
)

q18 = q18.sort_values("ReturnRatePct", ascending=False)

print("Q18 - RETURN RATE BY REGION")
display(q18)

Q18 - RETURN RATE BY REGION


,Region,EligibleOrders,ReturnedOrders,ReturnRatePct
0,East,1726,186,10.776362
3,West,2120,222,10.471698
1,North,2103,207,9.843081
2,South,2438,222,9.105824


In [ ]:
actual_monthly_region = (
    sales_items.assign(
        Month=sales_items["OrderDate"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby(["Month", "Region"], as_index=False)["Revenue"]
    .sum()
    .rename(columns={"Revenue": "ActualRevenue"})
)

q19 = targets.merge(
    actual_monthly_region,
    on=["Month", "Region"],
    how="left"
)

q19["ActualRevenue"] = q19["ActualRevenue"].fillna(0)
q19["Variance"] = q19["ActualRevenue"] - q19["SalesTarget"]

q19["AchievementPct"] = (
    q19["ActualRevenue"] / q19["SalesTarget"] * 100
)

q19_region = (
    q19.groupby("Region", as_index=False)
    .agg(
        TotalTarget=("SalesTarget", "sum"),
        ActualRevenue=("ActualRevenue", "sum"),
        MonthsAboveTarget=("AchievementPct", lambda x: (x >= 100).sum()),
        MonthsAnalysed=("Month", "nunique")
    )
)

q19_region["Variance"] = (
    q19_region["ActualRevenue"] - q19_region["TotalTarget"]
)

q19_region["AchievementPct"] = (
    q19_region["ActualRevenue"] / q19_region["TotalTarget"] * 100
)

q19_region = q19_region.sort_values(
    "AchievementPct", ascending=False
)

print("Q19 - REGIONAL TARGET PERFORMANCE")
display(q19_region)

Q19 - REGIONAL TARGET PERFORMANCE


,Region,TotalTarget,ActualRevenue,MonthsAboveTarget,MonthsAnalysed,Variance,AchievementPct
2,South,83042475.0,1.486034e+08,52,60,6.556089e+07,178.948621
3,West,86117369.0,1.250796e+08,47,60,3.896221e+07,145.243151
1,North,88995637.0,1.291071e+08,53,60,4.011145e+07,145.071257
0,East,86099721.0,1.060620e+08,39,60,1.996228e+07,123.185064


In [ ]:
print("Q20 - BUSINESS INSIGHTS")

print("Highest Revenue Region:", q4.iloc[0]["Region"])

print("Highest Revenue Product:", q5.iloc[0]["ProductName"])

print(
    "Top Employee:",
    employee_perf.sort_values("Revenue", ascending=False).iloc[0]["Name"]
)

print("Top Store:", q15.iloc[0]["StoreName"])

print("Top Sales Channel:", q16.iloc[0]["SalesChannel"])

print("Top Customer Segment:", q13.iloc[0]["CustomerSegment"])

print("Highest Return Rate Region:", q18.iloc[0]["Region"])

print(
    "Regions Below Annual Target:",
    q19_region.loc[
        q19_region["AchievementPct"] < 100, "Region"
    ].tolist()
)

Q20 - BUSINESS INSIGHTS
Highest Revenue Region: South
Highest Revenue Product: Premium Backpack 128
Top Employee: Aditya Sharma
Top Store: Kochi Market
Top Sales Channel: Online
Top Customer Segment: Regular
Highest Return Rate Region: East
Regions Below Annual Target: []


In [ ]:
results = {
    "q2_top_orders": q2,
    "q3_monthly_revenue": monthly_revenue,
    "q4_region_performance": q4,
    "q5_top_products": q5,
    "q6_top_categories": q6_categories,
    "q6_low_profit_products": q6_low_products,
    "q7_low_margin_products": q7,
    "q8_employee_performance": employee_perf,
    "q10_employee_product_sales": q10,
    "q11_best_employee_by_category": q11,
    "q12_top_customers": q12,
    "q13_customer_segments": q13,
    "q14_customers_no_orders": q14,
    "q15_store_performance": q15,
    "q16_sales_channels": q16,
    "q18_return_rate_region": q18,
    "q19_target_performance": q19_region
}

for name, frame in results.items():
    frame.to_csv(f"{name}.csv", index=False)

print("All result CSV files saved successfully!")

All result CSV files saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>